# Unitree Go2 Backflip Training on Google Colab

This notebook trains a PPO policy that makes the Unitree Go2 quadruped do a **backflip** in simulation, using Brax and MuJoCo MJX.

It reuses the setup of `train_unitree_go2_colab.ipynb` (same model, physics and PPO pipeline) but replaces the walking task with a flip task.

## Setup Instructions:
1. **Runtime → Change runtime type → A100 GPU** (T4 works but is several times slower)
2. Run all cells in order
3. Watch `rotation` and `flip_success` in the training printout (see section 8)
4. Download the trained policy from the Files panel

## How the task works:
- Each episode starts with the robot standing and lasts 2 seconds (100 control steps)
- **Flip phase:** reward for each bit of *new* backward rotation (rocking back and forth earns nothing) and for height once it has started rotating, until the body has turned a full circle (about 2π rad)
- **Landing phase:** one-off bonus for landing on its feet, then reward for staying upright and returning to the standing pose
- The episode ends early if the body hits the ground, the robot pitches forwards, or it falls over after landing
- Flip progress (integrated body pitch rate, like a gyro) and episode time are part of the observation, so the policy knows where it is in the move

⚠️ **A backflip is much harder to learn than walking.** The reward weights here are a starting point and may need tuning; see the tips at the end.

## 1. Install Dependencies

In [ ]:
# Brax's PPO trainer calls a JAX function removed in JAX 0.8, so pin JAX
# (and a Flax that supports it) instead of letting pip pull the newest.
!pip install mujoco mujoco_mjx brax "jax[cuda12]==0.7.2" "flax<0.12"
!pip install optax orbax-checkpoint
!pip install mediapy
!command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)

print("\n✅ All dependencies installed!")
print("⚠️ If Colab asks you to restart the session, do so and continue from section 2.")

## 2. Verify GPU and Configure Environment

In [ ]:
#@title Check GPU and Configure Environment

import subprocess
import os

# Check GPU
if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.')

# Add Nvidia EGL driver config for rendering
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{{
    "file_format_version" : "1.0.0",
    "ICD" : {{
        "library_path" : "libEGL_nvidia.so.0"
    }}
}}
""")

# Configure MuJoCo to use EGL rendering (GPU)
os.environ['MUJOCO_GL'] = 'egl'

# XLA optimization for better performance
xla_flags = os.environ.get('XLA_FLAGS', '')
xla_flags += ' --xla_gpu_triton_gemm_any=True'
os.environ['XLA_FLAGS'] = xla_flags

# Test MuJoCo installation
import mujoco
mujoco.MjModel.from_xml_string('<mujoco/>')

import jax
print(f"✅ GPU detected: {jax.devices()}")
print(f"✅ MuJoCo configured with EGL rendering")

## 3. Download MuJoCo Menagerie Models

In [ ]:
!git clone https://github.com/google-deepmind/mujoco_menagerie.git
print("\n✅ MuJoCo Menagerie downloaded!")

## 4. Import Libraries

In [ ]:
import functools
from typing import Any, Dict, List, Sequence, Tuple
from datetime import datetime
from etils import epath

import jax
from jax import numpy as jp
import numpy as np
import matplotlib.pyplot as plt
import mediapy as media
from IPython.display import HTML

from ml_collections import config_dict
from flax.training import orbax_utils
from orbax import checkpoint as ocp

import mujoco
from mujoco import mjx

from brax import base
from brax import envs
from brax import math
from brax.base import Motion, Transform
from brax.envs.base import PipelineEnv, State
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks
from brax.io import html, mjcf, model

# More legible numpy printing
np.set_printoptions(precision=3, suppress=True, linewidth=100)

print("✅ Libraries imported successfully!")

## 5. Define Unitree Go2 Backflip Environment

In [ ]:
#@title Unitree Go2 Backflip Environment Definition

GO2_ROOT_PATH = epath.Path('mujoco_menagerie/unitree_go2')


def get_config():
  """Returns reward config for the Unitree Go2 backflip environment."""

  def get_default_rewards_config():
    default_config = config_dict.ConfigDict(
        dict(
            scales=config_dict.ConfigDict(
                dict(
                    # --- flip phase (before a full rotation) ---
                    pitch_rate=2.0,       # new backward rotation per second
                    height=5.0,           # base height once rotating
                    # --- landing phase (after a full rotation) ---
                    land=500.0,           # one-off bonus for landing on its feet
                    upright=5.0,          # stay upright after landing
                    pose=-0.5,            # return to the standing pose
                    ang_vel=-0.05,        # stop spinning after the flip
                    # --- always on ---
                    torques=-0.0002,
                    action_rate=-0.005,
                    termination=-200.0,
                )
            ),
        )
    )
    return default_config

  default_config = config_dict.ConfigDict(
      dict(
          rewards=get_default_rewards_config(),
      )
  )

  return default_config


class UnitreeGo2BackflipEnv(PipelineEnv):
  """Environment for training a Unitree Go2 backflip policy in MJX.

  Each episode starts standing and lasts `episode_steps` control steps. The
  policy is rewarded for pitching backwards and gaining height until it has
  rotated a full turn, then for landing on its feet and standing still.
  """

  def __init__(
      self,
      obs_noise: float = 0.02,
      episode_steps: int = 100,
      scene_file: str = 'scene_mjx.xml',
      **kwargs,
  ):
    path = GO2_ROOT_PATH / scene_file
    sys = mjcf.load(path.as_posix())
    self._dt = 0.02  # this environment is 50 fps
    sys = sys.tree_replace({'opt.timestep': 0.004})
    # Keep the Menagerie actuator gains (kp=50) and torque limits: a flip
    # needs every bit of torque the motors have.

    n_frames = kwargs.pop('n_frames', int(self._dt / sys.opt.timestep))
    super().__init__(sys, backend='mjx', n_frames=n_frames)

    self.reward_config = get_config()
    # set custom from kwargs
    for k, v in kwargs.items():
      if k.endswith('_scale'):
        self.reward_config.rewards.scales[k[:-6]] = v

    torso_body = 'base'
    self._torso_idx = mujoco.mj_name2id(
        sys.mj_model, mujoco.mjtObj.mjOBJ_BODY.value, torso_body
    )
    if self._torso_idx == -1:
      raise ValueError(
          f'Body {torso_body!r} not found in Unitree Go2 MJCF. '
          'Update the torso body name to match the model.'
      )
    self._obs_noise = obs_noise
    self._episode_steps = episode_steps
    home_qpos = np.array(sys.mj_model.keyframe('home').qpos)
    self._init_q = jp.array(home_qpos)
    self._default_pose = home_qpos[7:]
    self._stand_height = home_qpos[2]

    # Hip, thigh and calf get different action ranges: the thighs and calves
    # need large swings to jump and tuck, the hips only small corrections.
    self._action_scale = jp.array([0.3, 1.0, 0.8] * 4)
    # Clip targets to the model's control ranges (front and back thighs differ).
    self.lowers = jp.array(sys.mj_model.actuator_ctrlrange[:, 0])
    self.uppers = jp.array(sys.mj_model.actuator_ctrlrange[:, 1])

    # Foot sites for Go2
    feet_site = [
        'FL_foot',
        'FR_foot',
        'RL_foot',
        'RR_foot',
    ]
    feet_site_id = [
        mujoco.mj_name2id(sys.mj_model, mujoco.mjtObj.mjOBJ_SITE.value, f)
        for f in feet_site
    ]
    assert not any(id_ == -1 for id_ in feet_site_id), 'Site not found.'
    self._feet_site_id = np.array(feet_site_id)
    self._foot_radius = 0.0175  # Go2 foot radius
    self._nv = sys.nv

  def reset(self, rng: jax.Array) -> State:
    rng, key = jax.random.split(rng)

    # small random perturbation of the standing pose
    qpos = self._init_q.at[7:].add(
        jax.random.uniform(key, (12,), minval=-0.05, maxval=0.05)
    )
    pipeline_state = self.pipeline_init(qpos, jp.zeros(self._nv))

    state_info = {
        'rng': rng,
        'last_act': jp.zeros(12),
        'pitch': 0.0,       # accumulated backward rotation [rad]
        'max_pitch': 0.0,   # furthest backward rotation so far [rad]
        'landed': False,
        'rewards': {k: 0.0 for k in self.reward_config.rewards.scales.keys()},
        'step': 0,
    }

    obs = self._get_obs(pipeline_state, state_info)
    reward, done = jp.zeros(2)
    # Brax's evaluator sums metrics over an episode, so log per-step values:
    # 'rotation' sums to the total rotation [rad] and 'flip_success' to 1 if
    # the robot landed a flip.
    metrics = {'rotation': 0.0, 'flip_success': 0.0}
    for k in state_info['rewards']:
      metrics[k] = state_info['rewards'][k]
    state = State(pipeline_state, obs, reward, done, metrics, state_info)
    return state

  def step(self, state: State, action: jax.Array) -> State:
    rng, _ = jax.random.split(state.info['rng'])

    # physics step
    motor_targets = self._default_pose + action * self._action_scale
    motor_targets = jp.clip(motor_targets, self.lowers, self.uppers)
    pipeline_state = self.pipeline_step(state.pipeline_state, motor_targets)
    x, xd = pipeline_state.x, pipeline_state.xd

    joint_angles = pipeline_state.q[7:]
    base_height = x.pos[self._torso_idx - 1, 2]
    up = jp.array([0.0, 0.0, 1.0])
    up_z = jp.dot(math.rotate(up, x.rot[self._torso_idx - 1]), up)
    local_angvel = math.rotate(xd.ang[0], math.quat_inv(x.rot[0]))

    # Integrate the body pitch rate, like a gyro would. A backflip (nose up,
    # then over backwards) is a negative rotation about the body y axis.
    pitch = state.info['pitch'] - local_angvel[1] * self.dt
    # Only rotation past the furthest point so far counts, so rocking back and
    # forth earns nothing.
    max_pitch = jp.maximum(state.info['max_pitch'], pitch)
    new_rotation = max_pitch - state.info['max_pitch']
    rotated = pitch > 2 * jp.pi - 0.3  # close enough to a full turn

    # foot contact data based on z-position
    foot_pos = pipeline_state.site_xpos[self._feet_site_id]
    contact = foot_pos[:, 2] - self._foot_radius < 2e-2
    on_feet = (jp.sum(contact) >= 3) & (up_z > 0.8) & (base_height > 0.15)
    just_landed = rotated & on_feet & ~state.info['landed']
    landed = state.info['landed'] | just_landed

    # done if the body hits the ground, it flips forwards, or it falls over
    # after landing
    done = base_height < 0.12
    done |= pitch < -1.0
    done |= landed & (up_z < 0.5)
    timeout = state.info['step'] + 1 >= self._episode_steps

    # reward
    flipping = ~rotated
    rewards = {
        'pitch_rate': jp.clip(new_rotation / self.dt, 0.0, 15.0) * flipping,
        # height only counts once the robot has started to rotate, so plain
        # hopping doesn't pay
        'height': jp.clip(base_height - self._stand_height, 0.0, 0.6)
        * flipping * (max_pitch > 0.3),
        'land': jp.float32(just_landed),
        'upright': jp.float32(landed & on_feet),
        'pose': jp.sum(jp.abs(joint_angles - self._default_pose)) * landed,
        'ang_vel': jp.sum(jp.square(xd.ang[0])) * landed,
        'torques': self._reward_torques(pipeline_state.qfrc_actuator),
        'action_rate': self._reward_action_rate(action, state.info['last_act']),
        'termination': jp.float32(done),
    }
    rewards = {
        k: v * self.reward_config.rewards.scales[k] for k, v in rewards.items()
    }
    reward = sum(rewards.values()) * self.dt

    state.metrics['rotation'] = pitch - state.info['pitch']
    state.metrics['flip_success'] = jp.float32(just_landed)
    state.metrics.update(rewards)

    # state management
    state.info['last_act'] = action
    state.info['pitch'] = pitch
    state.info['max_pitch'] = max_pitch
    state.info['landed'] = landed
    state.info['rewards'] = rewards
    state.info['step'] += 1
    state.info['rng'] = rng


    # Brax's auto-reset only restores the physics state and observation, so
    # clear our own episode bookkeeping when the episode ends.
    reset = done | timeout
    state.info['step'] = jp.where(reset, 0, state.info['step'])
    state.info['pitch'] = jp.where(reset, 0.0, state.info['pitch'])
    state.info['max_pitch'] = jp.where(reset, 0.0, state.info['max_pitch'])
    state.info['landed'] = jp.where(reset, False, state.info['landed'])
    state.info['last_act'] = jp.where(reset, 0.0, state.info['last_act'])

    obs = self._get_obs(pipeline_state, state.info)
    done = jp.float32(reset)
    state = state.replace(
        pipeline_state=pipeline_state, obs=obs, reward=reward, done=done
    )
    return state

  def _get_obs(
      self,
      pipeline_state: base.State,
      state_info: dict[str, Any],
  ) -> jax.Array:
    inv_torso_rot = math.quat_inv(pipeline_state.x.rot[0])
    local_rpyrate = math.rotate(pipeline_state.xd.ang[0], inv_torso_rot)
    local_vel = math.rotate(pipeline_state.xd.vel[0], inv_torso_rot)

    obs = jp.concatenate([
        local_rpyrate * 0.25,                                # body rates
        math.rotate(jp.array([0, 0, -1]), inv_torso_rot),    # projected gravity
        local_vel * 0.5,                                     # body velocity
        jp.array([state_info['pitch'] / (2 * jp.pi)]),       # flip progress
        jp.array([state_info['step'] / self._episode_steps]),  # episode time
        jp.array([jp.float32(state_info['landed'])]),        # landed flag
        pipeline_state.q[7:] - self._default_pose,           # motor angles
        pipeline_state.qd[6:] * 0.05,                        # motor speeds
        state_info['last_act'],                              # last action
    ])

    # clip, noise
    obs = jp.clip(obs, -100.0, 100.0) + self._obs_noise * jax.random.uniform(
        state_info['rng'], obs.shape, minval=-1, maxval=1
    )
    return obs

  # ------------ reward functions----------------
  def _reward_torques(self, torques: jax.Array) -> jax.Array:
    return jp.sqrt(jp.sum(jp.square(torques))) + jp.sum(jp.abs(torques))

  def _reward_action_rate(
      self, act: jax.Array, last_act: jax.Array
  ) -> jax.Array:
    return jp.sum(jp.square(act - last_act))

  def render(
      self, trajectory: List[base.State], camera: str | None = None,
      width: int = 240, height: int = 320,
  ) -> Sequence[np.ndarray]:
    camera = camera or 'track'
    return super().render(trajectory, camera=camera, width=width, height=height)


envs.register_environment('unitree_go2_backflip', UnitreeGo2BackflipEnv)
print("✅ UnitreeGo2BackflipEnv registered!")

## 6. Domain Randomization Function

In [ ]:
#@title Domain Randomization Function

def domain_randomize(sys, rng):
  """Randomizes the mjx.Model for better sim-to-real transfer."""
  @jax.vmap
  def rand(rng):
    _, key = jax.random.split(rng, 2)
    # friction randomization
    friction = jax.random.uniform(key, (1,), minval=0.6, maxval=1.4)
    friction = sys.geom_friction.at[:, 0].set(friction)
    # actuator randomization
    _, key = jax.random.split(key, 2)
    gain_range = (-5, 5)
    param = jax.random.uniform(
        key, (1,), minval=gain_range[0], maxval=gain_range[1]
    ) + sys.actuator_gainprm[:, 0]
    gain = sys.actuator_gainprm.at[:, 0].set(param)
    bias = sys.actuator_biasprm.at[:, 1].set(-param)
    return friction, gain, bias

  friction, gain, bias = rand(rng)

  in_axes = jax.tree_util.tree_map(lambda x: None, sys)
  in_axes = in_axes.tree_replace({
      'geom_friction': 0,
      'actuator_gainprm': 0,
      'actuator_biasprm': 0,
  })

  sys = sys.tree_replace({
      'geom_friction': friction,
      'actuator_gainprm': gain,
      'actuator_biasprm': bias,
  })

  return sys, in_axes


print("✅ Domain randomization function defined!")

## 7. Create Environment

In [ ]:
# Create environment
env_name = 'unitree_go2_backflip'
env = envs.get_environment(env_name)

print(f"✅ Environment created!")
print(f"  - Observation size: {env.observation_size}")
print(f"  - Action size: {env.action_size}")

## 8. Train Policy with Progress Tracking

Every evaluation prints three numbers to watch:
- **rotation**: average total backward rotation per episode in radians. A full flip is about 6.3.
- **flip_success**: fraction of evaluation episodes where the robot landed a flip on its feet.
- **reward**: total episode reward.

Typically `rotation` climbs first (the robot learns to rear up, then to throw itself backwards) and `flip_success` only rises once it reaches about 6. If `rotation` stalls around 1-3, see the tuning tips at the end.

In [ ]:
#@title Train Policy with Progress Tracking

# Training configuration
NUM_TIMESTEPS = 200_000_000  # flips need more samples than walking
NUM_EVALS = 20
EPISODE_LENGTH = 100  # 2 s at 50 Hz, must match the env's episode_steps
NUM_ENVS = 8192  # Large batch for GPU efficiency
BATCH_SIZE = 256

# Progress tracking
x_data = []
rot_data = []
success_data = []
reward_data = []
times = [datetime.now()]

def progress(num_steps, metrics):
  """Callback to track and display training progress."""
  times.append(datetime.now())
  x_data.append(num_steps)
  rot_data.append(metrics['eval/episode_rotation'])
  success_data.append(metrics['eval/episode_flip_success'])
  reward_data.append(metrics['eval/episode_reward'])

  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
  ax1.plot(x_data, rot_data, 'b-o')
  ax1.axhline(2 * np.pi, color='gray', linestyle='--', label='full flip')
  ax1.set_xlim([0, NUM_TIMESTEPS * 1.05])
  ax1.set_xlabel('# environment steps')
  ax1.set_ylabel('rotation per episode [rad]')
  ax1.legend()
  ax1.grid(True, alpha=0.3)
  ax2.plot(x_data, success_data, 'g-o')
  ax2.set_xlim([0, NUM_TIMESTEPS * 1.05])
  ax2.set_ylim([0, 1.05])
  ax2.set_xlabel('# environment steps')
  ax2.set_ylabel('flip success rate')
  ax2.grid(True, alpha=0.3)
  fig.suptitle('Unitree Go2 Backflip Training')
  plt.show()

  print(f"Steps: {num_steps:,} | Reward: {reward_data[-1]:.2f} | "
        f"Rotation: {rot_data[-1]:.2f} rad | Success: {success_data[-1]:.0%}")

# Setup checkpoint saving
ckpt_path = epath.Path('/tmp/unitree_go2_backflip/ckpts')
ckpt_path.mkdir(parents=True, exist_ok=True)

def policy_params_fn(current_step, make_policy, params):
  """Save checkpoints during training."""
  orbax_checkpointer = ocp.PyTreeCheckpointer()
  save_args = orbax_utils.save_args_from_target(params)
  path = ckpt_path / f'{current_step}'
  orbax_checkpointer.save(path, params, force=True, save_args=save_args)
  print(f"💾 Checkpoint saved: {path}")

# Configure PPO training
make_networks_factory = functools.partial(
    ppo_networks.make_ppo_networks,
    policy_hidden_layer_sizes=(128, 128, 128, 128))

train_fn = functools.partial(
    ppo.train,
    num_timesteps=NUM_TIMESTEPS,
    num_evals=NUM_EVALS,
    reward_scaling=1,
    episode_length=EPISODE_LENGTH,
    normalize_observations=True,
    action_repeat=1,
    unroll_length=20,
    num_minibatches=32,
    num_updates_per_batch=4,
    discounting=0.98,  # longer horizon than walking: the landing comes ~1 s after take-off
    learning_rate=3.0e-4,
    entropy_cost=1e-2,
    num_envs=NUM_ENVS,
    batch_size=BATCH_SIZE,
    network_factory=make_networks_factory,
    randomization_fn=domain_randomize,
    policy_params_fn=policy_params_fn,
    seed=0,
    progress_fn=progress
)

# Reset environment and start training
env = envs.get_environment(env_name)
eval_env = envs.get_environment(env_name)

print(f"\n🚀 Starting training...")
print(f"  - Total steps: {NUM_TIMESTEPS:,}")
print(f"  - Parallel envs: {NUM_ENVS}\n")

make_inference_fn, params, _ = train_fn(
    environment=env,
    eval_env=eval_env
)

print(f'\n🎉 Training completed!')
print(f'  - Time to JIT: {times[1] - times[0]}')
print(f'  - Time to train: {times[-1] - times[1]}')

## 9. Plot Training Progress

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(x_data, rot_data, 'b-', linewidth=2)
ax1.axhline(2 * np.pi, color='gray', linestyle='--', label='full flip')
ax1.set_xlabel('Training Steps', fontsize=12)
ax1.set_ylabel('Rotation per episode [rad]', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)
ax2.plot(x_data, success_data, 'g-', linewidth=2)
ax2.set_ylim([0, 1.05])
ax2.set_xlabel('Training Steps', fontsize=12)
ax2.set_ylabel('Flip success rate', fontsize=12)
ax2.grid(True, alpha=0.3)
fig.suptitle('Unitree Go2 Backflip Training Progress', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Save Trained Policy

In [ ]:
# Save and reload params.
model_path = '/tmp/mjx_brax_unitree_go2_backflip_policy'
model.save_params(model_path, params)

print(f"✅ Final policy saved: {model_path}")
print(f"\n📦 To download:")
print(f"1. Click Files icon (📁) in left sidebar")
print(f"2. Navigate to {ckpt_path} or {model_path}")
print(f"3. Right-click → Download")

# Reload params and create inference function
params = model.load_params(model_path)
inference_fn = make_inference_fn(params)
jit_inference_fn = jax.jit(inference_fn)

print(f"\n✅ Policy reloaded and ready for visualization!")

## 11. Visualize Trained Policy

In [ ]:
#@title Visualize Trained Policy

# @markdown Number of flips to roll out back to back:
n_episodes = 3  #@param {type: "integer"}

# Create evaluation environment
eval_env = envs.get_environment(env_name)
jit_reset = jax.jit(eval_env.reset)
jit_step = jax.jit(eval_env.step)

rng = jax.random.PRNGKey(0)
rollout = []
pitch_traces = []
successes = 0

for ep in range(n_episodes):
  rng, reset_rng = jax.random.split(rng)
  state = jit_reset(reset_rng)
  rollout.append(state.pipeline_state)
  pitch = [0.0]
  landed = False
  for i in range(EPISODE_LENGTH):
    act_rng, rng = jax.random.split(rng)
    ctrl, _ = jit_inference_fn(state.obs, act_rng)
    state = jit_step(state, ctrl)
    rollout.append(state.pipeline_state)
    pitch.append(pitch[-1] + float(state.metrics['rotation']))
    landed |= bool(state.metrics['flip_success'])
    if state.done:
      break
  pitch_traces.append(pitch)
  successes += landed
  print(f"Episode {ep + 1}: rotation {pitch[-1]:.2f} rad, "
        f"{'landed ✅' if landed else 'no landing ❌'}")

print(f"\n✅ {successes}/{n_episodes} flips landed")

# Pitch progress over time
plt.figure(figsize=(8, 4))
for i, p in enumerate(pitch_traces):
  plt.plot(np.arange(len(p)) * eval_env.dt, p, label=f'episode {i + 1}')
plt.axhline(2 * np.pi, color='gray', linestyle='--', label='full flip')
plt.xlabel('time [s]')
plt.ylabel('backward rotation [rad]')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Render with MuJoCo renderer
render_every = 1  # flips are fast: keep every frame
media.show_video(
    eval_env.render(rollout[::render_every], camera='track'),
    fps=1.0 / eval_env.dt / render_every)

## 12. Alternative: Render with Brax HTML Renderer

In [ ]:
#@title Alternative: Render with Brax HTML Renderer

# This provides an interactive 3D visualization
HTML(html.render(
    eval_env.sys.tree_replace({'opt.timestep': eval_env.dt}),
    rollout
))

## 13. Package and Download All Results

In [ ]:
#@title Package and Download All Results

# Create zip file with all outputs
!zip -r /tmp/unitree_go2_backflip_results.zip {ckpt_path} {model_path}

print("\n📦 All results packaged: /tmp/unitree_go2_backflip_results.zip")
print("\n✅ Download from Files panel (📁)")
print("\nContains:")
print(f"  - Final policy: {model_path}")
print(f"  - Checkpoints: {ckpt_path}")
print(f"\n💡 Use these files with a local inference script!")

## Summary

### What was trained:
- **Robot**: Unitree Go2 quadruped from [MuJoCo Menagerie](https://github.com/google-deepmind/mujoco_menagerie/tree/main/unitree_go2)
- **Task**: Standing backflip, landing on all four feet
- **Algorithm**: PPO (Proximal Policy Optimization)
- **Training steps**: 200M
- **Parallel environments**: 8192 (GPU-accelerated with MJX)
- **Domain randomization**: Friction (0.6-1.4x) + Actuator parameters (±5 gain)

### Files generated:
1. **`/tmp/mjx_brax_unitree_go2_backflip_policy`** - Final trained policy
2. **`/tmp/unitree_go2_backflip/ckpts/`** - Intermediate checkpoints

### Differences from the Go2 walking notebook:
| | Walking | Backflip |
|---|---|---|
| Episode | 1000 steps (20 s) | 100 steps (2 s) |
| Ends when | body upside down, joint limits, low body | body hits ground (< 0.12 m), pitches forward, falls after landing |
| Rewards | velocity tracking, stay level | new backward rotation, height, landing bonus, upright after landing |
| Actuators | kp overridden to 35 | Menagerie default kp=50 and torque limits |
| Action scale | 0.3 rad for every joint | 0.3 hip / 1.0 thigh / 0.8 calf |
| Joint target limits | same for all legs | model control ranges (front and back thighs differ) |
| Observation | 15-step history, command | body rates, gravity, velocity, flip progress, time, landed flag |
| Push perturbations | yes | no |
| Discount | 0.97 | 0.98 |

### Tuning tips if it doesn't flip:
- **Rotation stalls at 1-3 rad** (it rears up and falls back): raise `pitch_rate` (e.g. 3-4) or `height`, or give more steps.
- **Rotation reaches ~6 but `flip_success` stays 0** (completes the turn, crashes on landing): raise `land` (e.g. 1000) or `termination` (e.g. -400) so crashing costs more than spinning earns.
- **It over-rotates**: landing on its feet already requires being upright; you can also lower the `pitch_rate` cap of 15 rad/s.
- **Jerky motion**: increase `action_rate` (e.g. -0.01) once it flips reliably, then fine-tune from a checkpoint.
- Reward weights can be overridden without editing the class: `envs.get_environment(env_name, land_scale=1000.0)`.

### Sim-to-real caution:
This is a simulation-only result. The observation includes body velocity, which a real Go2 can only estimate, and the policy has not been tested for real hardware. Don't run it on a physical robot without much more validation.